# Modelado y Política de Aprobación al 25%

**Proyecto de Administración Actuarial — FES Acatlán, UNAM · Semestre 2026-1**

Este notebook ejecuta el pipeline modular (ver `src/`) y muestra: comparativa de modelos, construcción del scorecard de puntos, validación out-of-time y la simulación de la regla de negocio del 25%.

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import pandas as pd
import numpy as np

from src import config, load_data, features, models, metrics, scorecard, approval

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

In [ ]:
summary = json.load(open(config.TABLES / 'resumen_metricas.json', encoding='utf-8'))
comp = pd.DataFrame(summary['comparativa_oot']).T[['KS','AUROC','Gini','LogLoss','Brier']]
comp

## Comparativa de modelos (validación out-of-time)

Se entrenó con préstamos resueltos emitidos antes de 2013-06 y se validó con préstamos resueltos emitidos entre 2013-06 y 2015-01 (vintages nunca vistos por el modelo).

In [ ]:
cv_comp = pd.DataFrame(summary['comparativa_cv']).T[['KS_cv','AUROC_cv','Gini_cv','Brier_cv']]
cv_comp.columns = ['KS (CV)','AUROC (CV)','Gini (CV)','Brier (CV)']
cv_comp

## Scorecard de puntos

Variables seleccionadas por Information Value y sin colinealidad:

In [ ]:
print(summary['scorecard_variables'])
print('\nMétricas OOT del scorecard:')
print(pd.Series(summary['scorecard']))

In [ ]:
pts = pd.read_csv(config.TABLES / 'scorecard_puntos.csv')
pts[['variable','tramo','puntos']].head(10)
# Nota: el score final = puntos_base + suma de puntos de atributos

## Regla de aprobación al 25%

El banco sólo puede originar el **25% del monto total solicitado**. Se aprueban las solicitudes de menor riesgo (score más alto) hasta llenar el presupuesto; después, cada aprobado se clasifica en **Bueno/Malo** con su desenlace real.

In [ ]:
aprob = summary['aprobacion']
aleat = summary['aprobacion_aleatorio']
todos = summary['aprobacion_todos']

resumen = pd.DataFrame([
    ['Scorecard (modelo)', aprob['solicitudes_aprobadas'], aprob['monto_aprobado'], aprob['aprobados_buenos'], aprob['aprobados_malos'], aprob['bad_rate_aprobados']],
    ['Aleatorio', aleat['solicitudes_aprobadas'], aleat['monto_aprobado'], aleat['aprobados_buenos'], aleat['aprobados_malos'], aleat['bad_rate_aprobados']],
    ['Aprobar todos', todos['solicitudes_aprobadas'], todos['monto_aprobado'], todos['aprobados_buenos'], todos['aprobados_malos'], todos['bad_rate_aprobados']],
], columns=['Política','Aprobados','Monto aprobado (USD)','Buenos','Malos','Tasa de mora aprobados'])
resumen

El **scorecard reduce la mora entre aprobados de 9.0% a 3.6%**, es decir 2.5x menos créditos incobrables con el mismo presupuesto del 25%.

In [ ]:
lift = pd.read_csv(config.TABLES / 'deciles_scorecard_oot.csv')
lift[['decile','n','bad_rate','lift']]